# Modeling — Random Forest & Logistic Regression
**TFG: Evaluación del riesgo de ciberseguridad en PYMES basada en modelos de Machine Learning explicables**

This notebook trains and evaluates two classification models to predict whether a CVE will be exploited in the wild.

Sections:
1. Load data and define features
2. Train/test split
3. Logistic Regression
4. Random Forest
5. Model comparison
6. Save best model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

os.makedirs('figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)

DATASET_PATH = '../data/processed/model_dataset.csv'
RANDOM_STATE = 42

## 1. Load Data and Define Features

In [ ]:
df = pd.read_csv(DATASET_PATH)

# is_ransomware_associated is excluded due to data leakage:
# it derives from CISA KEV, the same source as the target variable.
FEATURES = [col for col in df.columns if col not in ['exploited_in_wild', 'is_ransomware_associated']]
TARGET   = 'exploited_in_wild'

X = df[FEATURES]
y = df[TARGET].astype(int)

print(f'Features: {len(FEATURES)}')
print(f'Samples:  {len(X):,}')
print(f'Positive (exploited):     {y.sum():,}  ({y.mean()*100:.2f}%)')
print(f'Negative (not exploited): {(1-y).sum():,}  ({(1-y).mean()*100:.2f}%)')

## 2. Train / Test Split

Stratified split to preserve the class ratio in both sets.
80% training, 20% test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set:  {len(X_train):,} samples  ({y_train.sum():,} positive)')
print(f'Test set:      {len(X_test):,} samples  ({y_test.sum():,} positive)')

# StandardScaler for Logistic Regression (RF does not require scaling)
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 3. Logistic Regression

`class_weight='balanced'` compensates for the 1:198 class imbalance.
`max_iter=1000` ensures convergence on this dataset size.

In [ ]:
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)

lr.fit(X_train_scaled, y_train)
y_pred_lr    = lr.predict(X_test_scaled)
y_proba_lr   = lr.predict_proba(X_test_scaled)[:, 1]

print('LOGISTIC REGRESSION')
print('=' * 50)
print(classification_report(y_test, y_pred_lr, target_names=['Not exploited', 'Exploited']))

In [ ]:
# Cross-validation — 5 folds, stratified
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results_lr = cross_validate(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    X_train_scaled, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1', 'precision', 'recall'],
    return_train_score=False
)

print('Logistic Regression — 5-Fold Cross Validation')
for metric in ['roc_auc', 'f1', 'precision', 'recall']:
    scores = cv_results_lr[f'test_{metric}']
    print(f'  {metric:<12}  mean={scores.mean():.4f}  std={scores.std():.4f}')

## 4. Random Forest

`class_weight='balanced'` handles imbalance.
`n_estimators=200` gives stable results without excessive compute time.
`n_jobs=-1` uses all available CPU cores.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('RANDOM FOREST')
print('=' * 50)
print(classification_report(y_test, y_pred_rf, target_names=['Not exploited', 'Exploited']))

In [ ]:
cv_results_rf = cross_validate(
    RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1', 'precision', 'recall'],
    return_train_score=False
)

print('Random Forest — 5-Fold Cross Validation')
for metric in ['roc_auc', 'f1', 'precision', 'recall']:
    scores = cv_results_rf[f'test_{metric}']
    print(f'  {metric:<12}  mean={scores.mean():.4f}  std={scores.std():.4f}')

## 5. Model Comparison

In [ ]:
# ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_rf = roc_auc_score(y_test, y_proba_rf)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lr, tpr_lr, color='#5B8DB8', linewidth=2, label=f'Logistic Regression (AUC = {auc_lr:.4f})')
ax.plot(fpr_rf, tpr_rf, color='#E07B54', linewidth=2, label=f'Random Forest (AUC = {auc_rf:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('figures/10_roc_curves.png', bbox_inches='tight')
plt.show()

print(f'AUC-ROC  LR: {auc_lr:.4f}')
print(f'AUC-ROC  RF: {auc_rf:.4f}')

In [ ]:
# Precision-Recall Curves (more informative under class imbalance)
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_proba_lr)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_proba_rf)
ap_lr = average_precision_score(y_test, y_proba_lr)
ap_rf = average_precision_score(y_test, y_proba_rf)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_lr, prec_lr, color='#5B8DB8', linewidth=2, label=f'Logistic Regression (AP = {ap_lr:.4f})')
ax.plot(rec_rf, prec_rf, color='#E07B54', linewidth=2, label=f'Random Forest (AP = {ap_rf:.4f})')
ax.axhline(y=y_test.mean(), color='k', linestyle='--', linewidth=1, label=f'Baseline ({y_test.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('figures/11_precision_recall_curves.png', bbox_inches='tight')
plt.show()

print(f'Average Precision  LR: {ap_lr:.4f}')
print(f'Average Precision  RF: {ap_rf:.4f}')

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not exploited', 'Exploited'],
                yticklabels=['Not exploited', 'Exploited'])
    ax.set_title(title)
    ax.set_ylabel('True label')
    ax.set_xlabel('Predicted label')

plt.tight_layout()
plt.savefig('figures/12_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary comparison table
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'AUC-ROC':   [auc_lr, auc_rf],
    'Avg Precision': [ap_lr, ap_rf],
    'F1 (exploited)':   [
        f1_score(y_test, y_pred_lr, pos_label=1),
        f1_score(y_test, y_pred_rf, pos_label=1)
    ],
    'Precision (exploited)': [
        precision_score(y_test, y_pred_lr, pos_label=1),
        precision_score(y_test, y_pred_rf, pos_label=1)
    ],
    'Recall (exploited)': [
        recall_score(y_test, y_pred_lr, pos_label=1),
        recall_score(y_test, y_pred_rf, pos_label=1)
    ],
}).set_index('Model').round(4)

print('MODEL COMPARISON')
print('=' * 60)
print(summary.to_string())

In [ ]:
# Random Forest feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 9))
importances.plot(kind='barh', ax=ax, color='#5B8DB8')
ax.set_title('Random Forest — Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('figures/13_feature_importances.png', bbox_inches='tight')
plt.show()

## 6. Save Best Model

The best model and the scaler are saved to `models/` for use in the SHAP notebook.

In [ ]:
joblib.dump(rf,     '../models/random_forest.pkl')
joblib.dump(lr,     '../models/logistic_regression.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Save test set for SHAP notebook
X_test_df = pd.DataFrame(X_test, columns=FEATURES)
X_test_df.to_csv('../data/processed/X_test.csv', index=False)
pd.Series(y_test.values, name='exploited_in_wild').to_csv('../data/processed/y_test.csv', index=False)

print('Saved:')
print('  models/random_forest.pkl')
print('  models/logistic_regression.pkl')
print('  models/scaler.pkl')
print('  data/processed/X_test.csv')
print('  data/processed/y_test.csv')